# Milestone 4: Statistical Inference & Analytical Modeling
**Project:** Public Health Data Visualization System  
**Dataset:** Global Health Statistics (1,000,000 records × 22 columns + engineered features from M2)  

**Objective:** Use statistical methods to validate the visual insights from Milestone 3 and build predictive models that explain and forecast health outcomes.

---
**Assigned to:** *(Abby, Asami, Annbel, Christine, Chrystabel, Esther, Joyce, Rita, Sharon)*  
**Branch:** `name/milestone4-task`

> **Before starting:** Run `load_dataset.ipynb` first, then run the Setup cell below.

---
## Collaboration & Pairing Notes

| Pair | M3 Section (Visual) | M4 Section (Statistical) | How They Connect |
|---|---|---|---|
| **Pair 1 — Trend & Map Specialists** | Section 3: Spatial & Temporal Exploration | Section 8: Trend Analysis & Uncertainty | M3 shows *where/when* trends happen visually; M4 models those trends and calculates margin of error |
| **Pair 2 — Core Relationships Team** | Section 2: Distribution & Relationship Mapping | Section 6: Correlation & Regression | M3 shows the visual scatter cloud; M4 calculates the line of best fit and R-squared |
| **Pair 3 — Design & Logic Validators** | Section 1: Visual Strategy & Comparative Design | Section 5: Hypothesis Testing & Significance | M3 compares two groups visually; M4 runs the T-test to prove the difference is statistically significant |
| **Pair 4 — System Foundation Duo** | Section 4: Design Justification & Insight Reporting | Section 7: Data Splitting & Predictive Validation | M3 writes the final insight justification; M4 ensures models are honest via train/test split |
| **Solo — Quality Controller** | Evolutionary Failure Log | Evolutionary Failure Log | Interviews all pairs: *What model failed? Which M3 insight did not survive statistical testing?* |

> **Rule:** Every section must use `df` loaded from `data/processed/global_health_enriched.csv.gz`. Do not reload the raw CSV.

---
## Setup & Data Loading
**Owned by:** All members — run this first before any section.

**What to do here:**
- Load the processed dataset from Milestone 2 (`global_health_enriched.csv.gz`)
- Import all statistical and modeling libraries: `numpy`, `pandas`, `scipy`, `statsmodels`, `sklearn`
- Confirm all M2 engineered features are present: `Severity_Index`, `DALY_Intensity`, `Avg_Incidence_Disease`, `High_Risk_Demographic`, `Vaccine_Available_Flag`, `Mortality_YoY_Change`, `Weighted_Time_Impact`, `Demographic_encoded`, `decade`, `Gender_Encoded`, `Disease_Category_Encoded`
- Define the `TARGET` variable (`Mortality Rate (%)`) and the `FEATURES` list used across all modeling sections
- Drop rows with missing values in model inputs and print the final model dataframe shape

**Deliverable check:** `df` and `model_df` loaded here are the single source of truth. All sections read from these variables only.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path

# ── Load M2 processed output ──────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'global_health_enriched.csv.gz'

df = pd.read_csv(PROCESSED_PATH, compression='gzip')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# ── Confirm M2 engineered features are present ────────────────────────────────
M2_FEATURES = [
    'Severity_Index', 'DALY_Intensity', 'Avg_Incidence_Disease',
    'High_Risk_Demographic', 'Vaccine_Available_Flag', 'Mortality_YoY_Change',
    'Weighted_Time_Impact', 'Demographic_encoded', 'decade',
    'Gender_Encoded', 'Disease_Category_Encoded'
]
missing = [f for f in M2_FEATURES if f not in df.columns]
if missing:
    print(f'WARNING — Missing M2 features: {missing}')
    print('Re-run milestone2_data_processing_transformation.ipynb first.')
else:
    print('✓ All M2 engineered features confirmed.')

# ── Define TARGET and FEATURES used across all modeling sections ───────────────
TARGET = 'Mortality Rate (%)'

FEATURES = [
    'Per Capita Income (USD)',
    'Healthcare Access (%)',
    'Doctors per 1000',
    'Hospital Beds per 1000',
    'Education Index',
    'Urbanization Rate (%)',
    'Severity_Index',          # M2 engineered
    'DALY_Intensity',          # M2 engineered
    'Disease_Category_Encoded' # M2 engineered
]

# ── Build model_df: drop rows with missing values in model inputs ──────────────
model_df = df[[TARGET] + FEATURES + ['Year', 'Country', 'Disease Category', 'decade']].copy()
model_df = model_df.dropna(subset=[TARGET] + FEATURES)
model_df = model_df.reset_index(drop=True)


print("\n" + "=" * 150)
print(f'model_df shape: {model_df.shape[0]:,} rows × {model_df.shape[1]} columns')

Loaded: 1,000,000 rows × 33 columns
✓ All M2 engineered features confirmed.

model_df shape: 1,000,000 rows × 14 columns


---
## 5. Hypothesis Testing & Significance
**Owned by:** Pair 3 — Design & Logic Validators  
**M3 Dependency → Section 1 (Visual Strategy & Comparative Design):** The two groups you visually separated in M3 Section 1 (e.g., high-mortality vs. low-mortality countries) are the exact groups you test here. You are taking the visual hunch from M3 and proving it with math.

**What to do here:**
- Define a clear null hypothesis and alternative hypothesis based on the M3 Section 1 visual comparison
- Split `model_df` into the same two groups identified in M3 (e.g., top-10 vs. bottom-10 countries by `Mortality Rate (%)`)
- Run an **independent samples T-test** (`scipy.stats.ttest_ind`) on `Mortality Rate (%)` between the two groups
- If comparing more than two groups (e.g., all Disease Categories), run a one-way **ANOVA** instead
- Report the **t-statistic**, **p-value**, and **95% confidence interval** for the difference in means
- Use `Severity_Index` or `DALY_Intensity` (M2 features) as optional secondary test variables
- State clearly whether you reject or fail to reject the null hypothesis

**Deliverable check:** p-value, confidence interval, and a plain-English conclusion must all be printed. If p > 0.05, move the finding to the Evolutionary Failure Log.

In [3]:
# Section 5: Define groups from M3 Section 1 and run T-test or ANOVA


In [4]:
# Section 5: Calculate and print confidence intervals and effect size


---
## 6. Correlation & Multi-Variable Regression
**Owned by:** Pair 2 — Core Relationships Team  
**M3 Dependency → Section 2 (Distribution & Relationship Mapping):** The scatter plot in M3 Section 2 showed a visual relationship between income, healthcare access, and mortality. This section formalises that relationship into a predictive equation with coefficients and R-squared.

**What to do here:**
- Compute a **correlation matrix** for `TARGET` and all `FEATURES` — use `Severity_Index` and `DALY_Intensity` (M2 features) as additional candidate predictors
- Visualise the correlation matrix as a heatmap
- Identify the strongest positive and negative correlations with `Mortality Rate (%)`
- Use correlation results to select the final predictor set for regression
- Fit a **Multiple Linear Regression** model using `statsmodels.OLS` (gives p-values per coefficient)
- Report: coefficients, p-values per feature, overall R-squared, and adjusted R-squared
- Interpret which variables most strongly drive `Mortality Rate (%)`

**Deliverable check:** The regression summary table must be printed. At least one M2 engineered feature must be tested as a predictor. Any feature with p > 0.05 must be noted.

In [5]:
# Section 6: Correlation matrix and heatmap


In [6]:
# Section 6: Multiple Linear Regression model — fit, summary, and interpretation


---
## 7. Data Splitting & Predictive Validation
**Owned by:** Pair 4 — System Foundation Duo  
**M3 Dependency → Section 4 (Design Justification & Insight Reporting):** The insights documented in M3 Section 4 are only trustworthy if the model that supports them performs well on unseen data. This section is the honesty check — if the model fails here, the M3 insight must be flagged in the Evolutionary Failure Log.

**What to do here:**
- Split `model_df` into **70% training / 30% testing** using `sklearn.model_selection.train_test_split` with `random_state=42`
- Print the shape of both splits to confirm the ratio
- Train a `LinearRegression` model on the training set using the same `FEATURES` from Setup
- Generate predictions on the **test set only**
- Evaluate with: **Mean Squared Error (MSE)**, **Root Mean Squared Error (RMSE)**, and **R-squared**
- Compare train R-squared vs. test R-squared — a large gap indicates overfitting
- Optionally use `Severity_Index` or `Disease_Category_Encoded` (M2 features) as additional predictors and compare performance

**Deliverable check:** Train and test metrics must both be printed side by side. If test R-squared < 0.5, flag the model as unreliable in the Evolutionary Failure Log.

In [13]:
# Section 7: Split data 70/30 and print split shapes

X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("=" * 150)
print(f'Training set : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)')
print("=" * 150)
print(f'Test set     : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)')
print("=" * 150)

Training set : 700,000 rows (70%)
Test set     : 300,000 rows (30%)


In [14]:
# Section 7: Train model on training set, predict on test set, report MSE, RMSE, R-squared

model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

train_r2   = r2_score(y_train, y_train_pred)
test_r2    = r2_score(y_test,  y_test_pred)
test_mse   = mean_squared_error(y_test, y_test_pred)
test_rmse  = np.sqrt(test_mse)

print(f'Train R² : {train_r2:.4f}')
print("=" * 150)
print(f'Test  R² : {test_r2:.4f}')
print("=" * 150)
print(f'Test  MSE : {test_mse:.4f}')
print("=" * 150)
print(f'Test  RMSE: {test_rmse:.4f}')
print("=" * 150)

if abs(train_r2 - test_r2) > 0.1:
    print('\nWARNING — Large train/test R² gap. Possible overfitting. Log in Evolutionary Failure Log.')
elif test_r2 < 0.5:
    print('\nWARNING — Test R² < 0.5. Model is unreliable. Log in Evolutionary Failure Log.')
else:
    print('\n✓ Model performance acceptable.')

Train R² : 0.0378
Test  R² : 0.0381
Test  MSE : 7.8627
Test  RMSE: 2.8040

WARNING — Test R² < 0.5. Model is unreliable. Log in Evolutionary Failure Log.


---
## 8. Trend Analysis & Uncertainty Interpretation
**Owned by:** Pair 1 — Trend & Map Specialists  
**M3 Dependency → Section 3 (Spatial & Temporal Exploration):** The time-series chart in M3 Section 3 showed a visual trend over years. This section models that trend mathematically, forecasts future values, and adds shaded confidence ribbons to show the margin of error.

**What to do here:**
- Aggregate `model_df` by `Year` to get the mean `Mortality Rate (%)` per year
- Use the `decade` column (M2 engineered feature) to annotate structural shifts in the trend
- Fit a **simple linear trend model** (OLS regression of `Mortality Rate (%)` on `Year`)
- Forecast values for 2–3 years beyond the dataset range
- Plot the historical trend, the fitted line, and the **95% confidence interval** as a shaded ribbon
- Optionally use `Mortality_YoY_Change` (M2 engineered feature) to annotate years with the largest shifts
- Write a plain-English explanation of what the margin of error means for the forecast

**Deliverable check:** The confidence ribbon must be visible on the plot. The forecast must extend beyond the last data year. A written interpretation of uncertainty must follow the chart.

In [9]:
# Section 8: Aggregate yearly trend and fit linear trend model


In [10]:
# Section 8: Plot trend with forecast and shaded 95% confidence interval ribbon


---
## Evolutionary Failure Log
**Owned by:** Solo — Quality Controller  
**Responsibility:** Interview all four pairs after they complete their sections. Ask: *"Which statistical test failed? Which M3 insight did not survive validation?"*

**What to document here:**
- Which M3 visual insight was not supported by the p-value in M4?
- Where did the regression model disagree with a visual hypothesis from M3?
- Which model had a test R-squared too low to be trusted?
- Which M2 engineered feature added no predictive value and should be dropped?
- What should the team revisit before final reporting in M5/M6?

This log is **non-negotiable** — it preserves both successes and failed experiments and ensures the team does not report misleading findings.

### Failure Log — Fill in after all sections are complete

| # | What Failed / Was Statistically Insignificant | M3 Section | M4 Section | Reason | Action |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |